## Cell type marker genes identification


In [1]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    # library(Libra)
    library(harmony)
    library(ComplexHeatmap)
    library(RColorBrewer)
    })
# set large memory  
options(future.globals.maxSize = 122*1024^3)

In [ ]:
## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## Reloading 
# Convert h5ad to h5seurat
# h5ad_file="./Results/Revision_2/clean_object_revision_03032026.h5ad"
# clean = schard::h5ad2seurat(h5ad_file)/
clean = readRDS("./Results/Revision_2/clean_object_vascular_cell_types.rds")
print(clean)

## QC check up
## Figure 1

In [ ]:
## organizing the meta data 
meta = clean@meta.data
meta$cell.class = meta$Cell_class
table(meta$cell.class)

clean@meta.data = meta

In [ ]:
# plot
library(tidyverse)
library(viridis)
library(hrbrthemes)

meta <- clean@meta.data
p <- list()
for (i in 1:length(unique(meta$cell.class))){
   p[[i]] <- meta[meta$cell.class == unique(meta$cell.class)[i],]%>%
            ggplot( aes(x=nFeature_RNA, color=cell.class, fill=cell.class)) +
              geom_histogram() +
              scale_fill_viridis(discrete=TRUE) +
              scale_color_viridis(discrete=TRUE) +
              theme_ipsum() +
              theme(
                legend.position="none",
                panel.spacing = unit(0.1, "lines"),
                strip.text.x = element_text(size = 8)
              ) +
              xlab("Number of feature") +
              ylab("") +
              ggtitle(unique(meta$cell.class)[i]) 
}

In [ ]:
#######################################################
###----- Figure_1B: Object UMAP---------###
#######################################################
col1=c('#F06719','#08415C','#23767C','#155a66','#DC143C',"#00BFC4","#fcbe05","#A26DC2","#5b844d","#0072B2")
names(col1)=c('Astrocyte','Neuron','Ependymal_Cell','Epithelial_Cell','Microglia_Macrophage_T','Oligodendrocyte',"Endothelial","Mural_Cell","Fibroblast","OPC")

umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
## ploting
p1 <- DimPlot(clean, label = F, repel = TRUE, 
                reduction = "umap.harmony.denoised",group.by = "cell.class",cols=col1,raster = T,pt.size = 5,raster.dpi=c(3200,3200)) + umap_theme+ggtitle("Cell_class")+NoLegend()
p2 <- DimPlot(clean, label = T, repel = TRUE, 
                reduction = "umap.harmony.denoised",group.by = "cell.class",cols=col1,raster = T,pt.size = 2) + umap_theme+ggtitle("Cell_class")
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1B_cell_class_umap.pdf",
#     patchwork::wrap_plots(p1, ncol = 1),
#       scale = 1, width = 9, height = 9)
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1B_cell_class_umap_legend.pdf",
#     patchwork::wrap_plots(p2, ncol = 1),
#       scale = 1, width = 11, height = 9)

# options(repr.plot.width = 11,repr.plot.height = 9)
# p1
# p2

## Cell type proportion plot

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)

In [ ]:
# ── 0. Load region metadata ───────────────────────────────────────────────────
region_meta <- read.csv("./Data/region.csv", stringsAsFactors = FALSE) %>%
  mutate(
    Region_layer_1    = str_to_title(str_trim(Region_layer_1)),
    Region_layer_4 = str_trim(Region_layer_4),
    Region_abb = str_trim(Final_abb)
  ) %>%
  select(Region_layer_1, Region_layer_4, Region_abb)
#   region_meta

In [ ]:
## get meta data to work on
meta <- clean@meta.data
meta$region_name = str_to_title(str_trim(meta$region_name))
meta$brain_region <- paste(meta$regionID,meta$region_name, sep = "_")

meta[meta$region_name == "Mid Temporal Gyrus",]$region_name = "Middle Temporal Gyrus"

In [ ]:
# ── 1. Get the count table ────────────────────────────────────────────────────
counts <- ddply(meta, .(meta$Cell_class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class", "brain_region", "freq")

## addint the region_abb to the another column
df = unique(meta[,c("brain_region","region_abb")])
rownames(df) = df$brain_region
counts$region_abb = df[counts$brain_region, "region_abb"]
counts$brain_region = gsub("Mid Temporal Gyrus","Middle Temporal Gyrus",counts$brain_region)

# ── 2. Parse numeric order and abbreviation from brain_region (e.g. "1_HIP") ─
counts$order           <- as.numeric(str_split_fixed(counts$brain_region, "_", n = 2)[, 1])
counts$brain_region_plot <- str_split_fixed(counts$brain_region, "_", n = 2)[, 2]

In [ ]:
region_levels         <- unique(counts$brain_region_plot)
counts$brain_region_plot <- factor(counts$brain_region_plot, levels = region_levels)
counts$Region_layer_4    <- factor(counts$Region_layer_4,    levels = layer4_order)

In [ ]:
# ── 3. Color maps ─────────────────────────────────────────────────────────────
col1 <- c(
  Astrocyte              = "#F06719", Neuron              = "#08415C",
  Ependymal_Cell         = "#23767C", Epithelial_Cell     = "#155a66",
  Microglia_Macrophage_T = "#DC143C", Oligodendrocyte     = "#00BFC4",
  Endothelial            = "#fcbe05", Mural_Cell          = "#A26DC2",
  Fibroblast             = "#5b844d", OPC                 = "#0072B2"
)
counts$cell_class <- factor(counts$cell_class)
 
region_color_map <- c(
  "Cerebral Cortex"            = "#009E73",
  "Brain Stem and Spinal Cord" = "#D55E00",
  "Cerebellum"                 = "#046299",
  "Subcortical Region"         = "#03B8D8",
  "Cerebral Cortex Watershed"  = "#E69F00",
  "White Matter"               = "#CC79A7",
  "Olfactory Bulb"             = "#999999",
  "Choroid Plexus"             = "#9C0AE0",
  "Leptomeninges"              = "#915330",
  "Major Vessel"               = "#FF0000"
)
 
# ── 4. Lookup: region → layer4 color ─────────────────────────────────────────
rl4_lookup <- counts %>%
  select(brain_region_plot, Region_layer_4) %>%
  distinct() %>%
  mutate(brain_region_plot = as.character(brain_region_plot),
         Region_layer_4    = as.character(Region_layer_4))
 
tick_colors <- region_color_map[
  rl4_lookup$Region_layer_4[match(region_levels, rl4_lookup$brain_region_plot)]
]
 
# ── 5. Build dendrogram segments ─────────────────────────────────────────────
# Dendrogram is drawn top-down, then y-axis is reversed so it hangs below bars.
# y = 0  → top (connects to bar x-axis)
# y = 1  → horizontal grouping bar
# y = 2  → stem tip / label
region_x <- setNames(seq_along(region_levels), region_levels)
 
dend_segs  <- list()
dend_label <- list()
 
for (l4 in layer4_order) {
  members <- rl4_lookup %>% filter(Region_layer_4 == l4) %>% pull(brain_region_plot)
  if (length(members) == 0) next
 
  xs    <- region_x[members]
  x_min <- min(xs);  x_max <- max(xs);  x_mid <- (x_min + x_max) / 2
  col   <- region_color_map[[l4]]
 
  # Vertical ticks: top (y=0) → horizontal bar (y=1)
  for (xi in xs) {
    dend_segs[[length(dend_segs) + 1]] <- data.frame(
      x = xi, xend = xi, y = 0, yend = 1, col = col, stringsAsFactors = FALSE)
  }
  # Horizontal bar at y=1
  dend_segs[[length(dend_segs) + 1]] <- data.frame(
    x = x_min, xend = x_max, y = 1, yend = 1, col = col, stringsAsFactors = FALSE)
  # Vertical stem: y=1 → y=2
  dend_segs[[length(dend_segs) + 1]] <- data.frame(
    x = x_mid, xend = x_mid, y = 1, yend = 2, col = col, stringsAsFactors = FALSE)
 
  # Label at y=2 (will appear below stem after y-axis reversal)
  dend_label[[length(dend_label) + 1]] <- data.frame(
    x = x_mid, y = 2.08, label = l4, col = col, stringsAsFactors = FALSE)
}
 
dend_segs_df  <- bind_rows(dend_segs)
dend_label_df <- bind_rows(dend_label)
n_regions     <- length(region_levels)
 
# ── 6. Dendrogram panel (y reversed: y=0 at top, labels hang downward) ───────
p_dend <- ggplot() +
  geom_segment(data = dend_segs_df,
               aes(x = x, xend = xend, y = y, yend = yend, colour = I(col)),
               linewidth = 0.65) +
  geom_text(data = dend_label_df,
            aes(x = x, y = y, label = label, colour = I(col)),
            angle = 90, hjust = 1, vjust = 0.5,
            size = 2.7, fontface = "bold") +
  scale_x_continuous(limits = c(0.5, n_regions + 0.5), expand = c(0, 0)) +
  scale_y_reverse(limits = c(6, 0), expand = c(0, 0)) +
  theme_void() +
  theme(plot.margin = margin(0, 2, 2, 2))
 
# ── 7. Bar plot panel (no x-axis labels — dendrogram carries them) ────────────
p_bar <- ggplot(counts, aes(x = brain_region_plot_2, y = freq, fill = cell_class)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = col1, name = "Cell class") +
  scale_x_discrete(expand = c(0, 0.5)) +
  labs(x = NULL, y = "Cell count") +
  theme_bw(base_size = 10) +
  theme(
    axis.text.x        = element_text(angle = 90, hjust = 1, vjust = 0.5,
                                      size = 7, colour = tick_colors),
    axis.ticks.x       = element_line(colour = tick_colors),
    panel.grid.major.x = element_blank(),
    panel.grid.minor   = element_blank(),
    legend.position    = "bottom",
    legend.key.size    = unit(0.4, "cm"),
    plot.margin        = margin(5, 2, 0, 2)
  )
# ── 8. Stack: bar on top, dendrogram on bottom ────────────────────────────────
final_plot <- p_bar / p_dend +
  plot_layout(heights = c(4, 1.8))

ggsave("./Results/Revision_2/Figures/Figure_1X_cell_counts_with_dendrogram.pdf", plot = final_plot,
       width = 8, height = 5.2, bg = "white")
message("Saved: cell_counts_with_dendrogram.pdf")
final_plot

## For Supplementary figure 1: Proportional plot with sum=100%

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)
## get the count table
counts <- ddply(meta, .(meta$cell.class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])

In [ ]:
library(plyr)
library(ggplot2)
library(stringr)
## get the count table
counts <- ddply(meta, .(meta$cell.class, meta$brain_region), nrow)
colnames(counts) <- c("cell_class","brain_region","freq")

## reorder the brain region based on the number code
counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])
counts <- counts[order(counts$cell_class),]
counts <- counts[order(counts$order),]

## making sure corrected order
counts$brain_region_plot = str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,2]
counts$brain_region_plot<- factor(counts$brain_region_plot, levels=unique(counts$brain_region_plot))


counts$cell_class <- factor(counts$cell_class)
## colors
col1=c('#F06719','#08415C','#23767C','#155a66','#DC143C',"#00BFC4","#fcbe05","#A26DC2","#5b844d","#0072B2")
names(col1)=c('Astrocyte','Neuron','Ependymal_Cell','Epithelial_Cell','Microglia_Macrophage_T','Oligodendrocyte',"Endothelial","Mural_Cell","Fibroblast","OPC")
counts$color <- col1[counts$cell_class]
# counts

In [ ]:
# plotting the cell types
p3 <- ggplot(counts, aes(fill=cell_class, y=freq, x=brain_region_plot)) + 
        geom_bar(position="stack", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        scale_fill_manual(values=col1) +
        xlab("")

p4 <- ggplot(counts, aes(fill=cell_class, y=freq, x=brain_region_plot)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
              axis.text.y = element_text(size = 15),
              axis.title.y = element_text(size = 15)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 8, repr.plot.res = 100) # set the resolution
p3
p4
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1E_Cell_class_region_proportion.pdf", 
# patchwork::wrap_plots(p4, ncol = 1),
# scale = 1, width = 12, height = 7.5)

### Chi-sq test to check cell type enrichment

In [ ]:
### try chisq contigency table
meta <- clean@meta.data
meta$brain_region <- paste(meta$regionID,meta$region_abb,sep = "_")

obs_mat = as.matrix(table(meta$region_name,meta$cell.class))

In [ ]:
# run chisq test
chisq_res <- chisq.test(obs_mat)
exp_mat = chisq_res$expected

# get the log1p(mat)
res = (obs_mat/exp_mat)+1
res = log(res)
res = res[order(as.numeric(str_split_fixed(rownames(res),pattern = "_",n = 2)[,1])),]

In [ ]:
## plotting
library(ComplexHeatmap)
library(colorRamp2)

#options(repr.plot.width = 10, repr.plot.height = 12, repr.plot.res = 100) # set the resolution
p = Heatmap(res,col = colorRamp2(c(0,1.5,3),c("grey95","red","red4")),cluster_rows = T,cluster_columns = F)

pdf("./Results/Revision_2/Figures/Figure_S1_Cell_state_enrichment_cluster_ALL.pdf",width = 7,height = 11)
p
dev.off()

## Redo the normalization for cell type marker gene identification
#### check if the data is clean enough

In [ ]:
## set cell identity
DefaultAssay(clean) <- "decontX"
clean
Idents(clean) <- "Cell_class"
table(Idents(clean)) 
## normalization
clean <- NormalizeData(clean)
clean <- JoinLayers(clean)

In [ ]:
#### code for identifying the cell type makers ####
# the parameters are based on https://www.cell.com/cell/fulltext/S0092-8674(23)00973-X
table(Idents(clean))
# clean <- PrepSCTFindMarkers(clean)
# running differential analysis
all_markers_major <- FindAllMarkers(object = clean,verbose = T,min.pct = 0.25,logfc.threshold = 0.25
#,test.use = "MAST" #might use MAST model
) 
write.csv(all_markers_major,file = "./Results/Revision_2/DEG/All_cellclass_markers_Wilcoxon_0304.csv")

### Load the cell type DEG results

In [ ]:
######################################################################
###########-- check cell type marker genes on each cluster --#########
######################################################################
library(viridis)
Idents(clean) <- "Cell_class"
Idents(clean) <- factor(clean@active.ident, sort(levels(clean@active.ident),decreasing = T))

## set assay to RNA
DefaultAssay(clean) <- "decontX"

p1 <-  DotPlot(clean,cols = viridis(3, option = "magma"),
             features = c("SLC1A2","RYR3","AQP4","GFAP",#Astrocytes
                            "FLT1","MECOM","ATP10A",  # Endothelial
                            "CFAP54","CFAP46","HTR2C","ABCA4", # Ependymal_Epithelial
                            "CEMIP","BICC1","LAMA2", #Fibroblast
                            "C3","RYR1","DOCK2",# Microglia_Macrophage_T
                            "CARMN","MYH11","RGS5", #Mural cell
                            "SYT1","ROBO2","CNTNAP2", # Neuron
                            "MOBP","ST18", "SLC24A2",#OLI
                            "TNR","DSCAM","LHFPL3" # OPC
                                )) + RotatedAxis() +
                                    theme(axis.title = element_blank(),legend.position = "top")
                                    
ggsave(filename = "./Results/Revision_2/Figures/Figure_1C_cellclass_dot_plot.pdf",
   patchwork::wrap_plots(p1, ncol = 1),
     scale = 1, width = 8, height = 4)

# p1

## Save copy in h5ad format for other python based analysis

In [ ]:
## check if python is available and check which version is under using
## python packages ## wired issue
# reticulate::use_python("/oak/stanford/projects/kibr/Reorganizing/Projects/Andi/software/miniforge3/envs/sc/bin")
reticulate::py_discover_config()
#py_install("numpy")
reticulate::py_module_available("numpy")
reticulate::py_module_available("anndata")

In [ ]:
## check before subsetting
clean
options(repr.plot.width= 10)
DimPlot(clean, reduction = "umapharmony_",group.by = "Cell_type")

In [ ]:
DefaultAssay(clean) <- "RNA"
#integrated <- JoinLayers(integrated)
clean[["RNA"]] <- as(object = clean[["RNA"]], Class = "Assay")
clean[["RNA"]] # check the version of the seurat assay
sceasy::convertFormat(clean, from="seurat", to="anndata",
                       outFile='./Results/03.object_class_0620.h5ad')

## Plot more QC matrics

In [ ]:
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
p0 <- DimPlot(clean, reduction = "umapharmony_",group.by = "cell.class",label = T,raster = T,pt.size = 2)+umap_theme+NoLegend()
p1 <- DimPlot(clean, reduction = "umapharmony_",group.by = "individualID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p2 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sampleID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p3 <- DimPlot(clean, reduction = "umapharmony_",group.by = "brain_region",raster = T,pt.size = 2)+umap_theme+NoLegend()
p4 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sex",raster = T,pt.size = 2)+umap_theme+NoLegend()
p5 <- FeaturePlot(clean, reduction = "umapharmony_",features = "ageatdeath",cols = brewer.pal(n = 8, name = 'Blues'),raster = T,pt.size = 2)+umap_theme+NoLegend()

In [ ]:
# ggsave(filename = "./Results/Revision_2/Figures/Figure_S1_clean_object_QC.pdf",
#     patchwork::wrap_plots(p0,p1,p2,p3,p4,p5, ncol = 3),
#       scale = 1, width = 21, height = 14)

In [ ]:
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))
p0 <- DimPlot(clean, reduction = "umapharmony_",group.by = "cell.class",label = T,raster = T,pt.size = 2)+umap_theme+NoLegend()
p1 <- DimPlot(clean, reduction = "umapharmony_",group.by = "individualID",raster = T,pt.size = 2)+umap_theme
p2 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sampleID",raster = T,pt.size = 2)+umap_theme+NoLegend()
p3 <- DimPlot(clean, reduction = "umapharmony_",group.by = "brain_region",raster = T,pt.size = 2)+umap_theme+NoLegend()
p4 <- DimPlot(clean, reduction = "umapharmony_",group.by = "sex",raster = T,pt.size = 2)+umap_theme
p5 <- FeaturePlot(clean, reduction = "umapharmony_",features = "ageatdeath",cols = brewer.pal(n = 8, name = 'Blues'),raster = T,pt.size = 2)+umap_theme

In [ ]:
# ggsave(filename = "./Results/Revision_2/Figures/Figure_S1_clean_object_metadata_legend.pdf",
#     patchwork::wrap_plots(p0,p1,p2,p3,p4,p5, ncol = 3),
#       scale = 1, width = 24, height = 14)

### Draw ComplexHeatmap to show donor by region matrix of number of cells

In [ ]:
### Draw ComplexHeatmap to show donor by region matrix of number of cells
meta = clean@meta.data
mat = as.matrix(table(meta$individualID,meta$region_name))
mat

In [ ]:
## Provide the color code
meta <- clean@meta.data
df <- unique(meta[,c("region_name","region_layer",'regionID')])

df = as.data.frame(df)

df$region_color <- NA
df[df$region_layer == "Cerebral Cortex",]$region_color = "#009E73"
df[df$region_layer == "Brain Stem and Spinal Cord",]$region_color = "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color = "#0072B2"
df[df$region_layer == "Subcortical Region",]$region_color = "#56B4E9"
df[df$region_layer == "Cerebral Cortex Watershed",]$region_color = "#E69F00"
df[df$region_layer == "White Matter",]$region_color = "#CC79A7"
df[df$region_layer == "Olfactory Bulb",]$region_color = "#999999"
df[df$region_layer == "Choroid Plexus",]$region_color = "#F0E442"
df[df$region_layer == "Leptomeninges",]$region_color = "#000000"
df[df$region_layer == "Major Vessel",]$region_color = "#FF0000"

# ## Assign region color
region_color <- df$region_color
names(region_color) <-df$region_name

In [ ]:
mat = mat[,df[order(as.numeric(df$regionID)),]$region_name]
mat

In [ ]:
library(colorRamp2)
# annotation for heatmap
hl <- HeatmapAnnotation(
  region = colnames(mat),
#   avg_density = avg_vec,
  col = list(
    region = region_color
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(mat),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(mat))
)
# Main heatmap
ht <- Heatmap(
  mat,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = FALSE,
  show_heatmap_legend = TRUE,
  use_raster = TRUE,
  col = colorRamp2(c(0, 500, 2000, 5000, 15000), 
           c("#FFFFFF", "#C6DBEF", "#6BAED6", "#2171B5", "#08306B"))
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

In [ ]:
# pdf(file = "./Results/Revision_2/Figures/Figure_1X_number_of_cells_per_region.pdf", width = 13, height = 5)
# ht
# dev.off()
# ht

In [ ]:
## check and save the clean object for future use
clean
saveRDS(clean,file = "./Results/Revision_2/clean_object_vascular_cell_types.rds")